In [ ]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import scipy.interpolate as interpolate
import tqdm

import icepinn


paths = [pathlib.Path(f) for f in sorted(glob.glob('data/V6/*.nc'))]


In [ ]:
for i, p in enumerate(paths):
    print(f'Working on index {i} / file {p.name}...')
    try:
        mask = []
        seaice_conc_interp = []
        seaice_conc_stdev_interp = []

        data = icepinn.nc.SeaIceV6([p])

        xx, yy = np.meshgrid(data.x, data.y, indexing='ij')
        for j, day in enumerate(data.date):
            print(f'\tWorking on day {j}...', end='\r')

            day_seaice_conc = np.copy(data.seaice_conc[j])
            day_seaice_conc_stdev = np.copy(data.seaice_conc_stdev[j])
            day_mask = np.isnan(day_seaice_conc) | np.isnan(day_seaice_conc_stdev)            

            day_seaice_conc[day_mask] = interpolate.NearestNDInterpolator(
                np.stack((xx[~day_mask], yy[~day_mask]), axis=-1),
                day_seaice_conc[~day_mask]
            )(xx[day_mask], yy[day_mask])
            day_seaice_conc_stdev[day_mask] = interpolate.NearestNDInterpolator(
                np.stack((xx[~day_mask], yy[~day_mask]), axis=-1),
                day_seaice_conc_stdev[~day_mask]
            )(xx[day_mask], yy[day_mask])

            mask.append(day_mask)
            seaice_conc_interp.append(day_seaice_conc)
            seaice_conc_stdev_interp.append(day_seaice_conc_stdev)

        dest = p.parent / ('_'.join(p.stem.split('_')[2:]) + '_interp')
        np.savez_compressed(
            dest,
            date=data.date,
            x=data.x,
            y=data.y,
            latitude=data.latitude,
            longitude=data.longitude,
            mask=np.stack(mask),
            seaice_conc_interp=np.stack(seaice_conc_interp),
            seaice_conc_stdev_interp=np.stack(seaice_conc_stdev_interp),
        )
    except Exception as e:
        print(f'Failed on index {i} / file {p.name}:\n\t')
        raise e


In [ ]:
f = np.load('data/V6/19781025-19781231_v06r00_interp.npz')

plt.imshow(f['seaice_conc_interp'][0].T, cmap='Blues_r')

cmap = plt.get_cmap('Blues_r')
cmap.set_bad("#ff8d6b", alpha=0.5)
_s = f['seaice_conc_interp'][0]
_s[f['mask'][0]] = np.nan
plt.imshow(_s.T, cmap=cmap)

plt.show()


In [ ]:
files_npz = [pathlib.Path(f) for f in sorted(glob.glob('data/V6/*.npz'))]
global_mask = []
global_date = []
x, y = np.empty((0,)), np.empty((0,))

for i, f in enumerate(files_npz):
    print(
        f'Year = {i + 1978}',
        end='\r'
    )
    ff = np.load(f)
    global_mask.append(
        ff['mask']
    )
    global_date.append(
        ff['date']
    )

    if i == 0:
        x, y = ff['x'], ff['y']

np.savez_compressed(
    'data/V6/coord_global.npz',
    mask=np.concat(global_mask),
    date=np.concat(global_date),
    x=x,
    y=y
)
